# Deep-RL-Replenish: Multi-Hub Inventory Optimization via Deep RL

This exploration notebook walks through the **Markov Decision Process (MDP)** formulation of the quick-commerce multi-hub replenishment problem:
- **State Space $\mathcal{S}$**: On-hand inventory, lead-time pipeline shipments, rolling demand history, and perishable days-to-spoil.
- **Action Space $\mathcal{A}$**: Continuous replenishment orders across $H$ Dark Stores $\times$ $S$ SKUs.
- **Competing Pressures**: Holding costs vs. severe stockout penalties vs. fixed dispatch costs vs. perishable spoilage.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure repo root is in python path
root_dir = str(Path.cwd().parent)
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

from src.env import MultiHubReplenishmentEnv
from src.baselines import BaseStockPolicy, EOQPolicy
from src.fefo_bridge import FEFOCommercialBridge
from src.utils import load_config

print("Environment and baseline modules imported successfully.")

## 1. Environment Initialization & State Inspection
Let's instantiate the multi-hub environment configured with 3 Dark Stores (Delhi North, Gurgaon Cyber, Noida Sec 62) and 8 Core SKUs.

In [ ]:
env = MultiHubReplenishmentEnv(seed=42)
obs, _ = env.reset(seed=42)

print(f"Observation Space Shape : {env.observation_space.shape}")
print(f"Action Space Shape      : {env.action_space.shape}")
print(f"Number of Dark Stores   : {env.n_hubs}")
print(f"Number of SKUs          : {env.n_skus}")
print(f"Supplier Lead Time      : {env.lead_time} days")
print(f"Perishable SKUs         : {env.perishable_skus} (Shelf life: {env.shelf_life_days} days)")

## 2. Baseline Rollout: Classical (s, S) Base-Stock Policy
We simulate a 60-day operational quarter under classical Base-Stock policy with a 95% cycle service level target.

In [ ]:
base_stock = BaseStockPolicy(
    n_skus=env.n_skus,
    n_hubs=env.n_hubs,
    lead_time_days=env.lead_time,
    demand_mean=env.demand_mean,
    demand_std=env.demand_std,
    service_level=0.95,
)

obs, _ = env.reset(seed=100)
history = []
done = False

while not done:
    action = base_stock.predict(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    history.append({
        "day": info["day"],
        "cost_total": info["cost_total"],
        "cost_holding": info["cost_holding"],
        "cost_stockout": info["cost_stockout"],
        "cost_spoilage": info["cost_spoilage"],
        "service_level": info["service_level"] * 100.0,
        "on_hand": np.sum(env.on_hand),
        "demanded": info["units_demanded"],
        "served": info["units_served"],
    })
    done = terminated or truncated

df_base = pd.DataFrame(history)
print(f"60-Day Base-Stock Total Cost: ₹{df_base['cost_total'].sum():,.0f}")
print(f"Final Achieved Service Level: {df_base['service_level'].iloc[-1]:.1f}%")

## 3. Inventory & Cost Dynamics Visualization
Let's plot the daily network on-hand inventory trajectory and the decomposition of costs over the 60-day horizon.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(df_base["day"], df_base["on_hand"], color="#2563eb", label="Total On-Hand Inventory", lw=2)
ax1.plot(df_base["day"], df_base["demanded"], color="#ef4444", label="Daily Demand", linestyle="--", alpha=0.7)
ax1.set_ylabel("Units")
ax1.set_title("Base-Stock (s, S) 60-Day Trajectory", fontweight="bold")
ax1.legend()
ax1.grid(True, linestyle=":", alpha=0.6)

ax2.bar(df_base["day"], df_base["cost_holding"], label="Holding Cost", color="#3b82f6", alpha=0.7)
ax2.bar(df_base["day"], df_base["cost_stockout"], bottom=df_base["cost_holding"], label="Stockout Cost", color="#dc2626", alpha=0.7)
ax2.bar(df_base["day"], df_base["cost_spoilage"], bottom=df_base["cost_holding"] + df_base["cost_stockout"], label="Spoilage", color="#f59e0b", alpha=0.7)
ax2.set_xlabel("Operational Day")
ax2.set_ylabel("Daily Cost (₹)")
ax2.legend()
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

## 4. FEFO Bridge: Translating Policy Actions to Commercial POs
Real quick-commerce operations require supplier case-pack rounding, vendor Minimum Order Quantities (MOQ), and shelf-life damping from Fresh-Grade.

In [ ]:
bridge = FEFOCommercialBridge()
raw_orders = [
    {"hub_id": 0, "hub_name": "DarkStore-DEL-North", "sku_id": 0, "recommended_reorder_qty": 13.8},
    {"hub_id": 0, "hub_name": "DarkStore-DEL-North", "sku_id": 1, "recommended_reorder_qty": 4.2},
    {"hub_id": 0, "hub_name": "DarkStore-DEL-North", "sku_id": 2, "recommended_reorder_qty": 21.0},
]
df_raw = pd.DataFrame(raw_orders)
df_po = bridge.convert_actions_to_pos(df_raw, shelf_life_estimates={0: 3.5, 2: 2.0})

df_po[["sku_name", "vendor_name", "raw_rl_recommendation", "case_pack_size", "vendor_moq", "final_po_units", "total_po_value_inr", "approval_status"]]